## Rootless Docker — the alternative install

Module 08 introduced rootless Docker as the hardening move that runs the **entire daemon as your unprivileged user** instead of root. Here's the install side of it — a different path from the standard package install, run **as your normal user, never with `sudo`**:

```bash
curl -fsSL https://get.docker.com/rootless | sh
# then add ~/.bin to PATH and set DOCKER_HOST as the script prints

systemctl --user enable docker
systemctl --user start docker
docker info    # should show "Security Options: rootless"
```

Two things move compared to a rootful install: the daemon's state lives under **`~/.local/share/docker/`** (your home, not `/var/lib/docker`), and networking goes through **`slirp4netns`**, a userspace network stack, rather than host `iptables`.

The trade-offs are exactly the ones from module 08 — slower networking, no `--privileged`, no host `iptables` writes (so port publishing routes through `slirp4netns`), and cgroup v2 required for resource limits. What you buy is a **much smaller attack surface**: there is no root daemon to compromise, so a container escape gets an attacker only your own UID's privileges.

The decision rule is the same as before: for a single-owner dev laptop or a normal production server, the standard rootful install is fine (and faster). Reach for rootless on **shared dev hosts, CI runners, and multi-tenant sandboxes** — anywhere a compromised container or build step must not become a compromised host. It's the same daemon and the same `docker` commands; only *who owns the process* changes.